# Ноутбук 05 — Предсказание и submission-файл

**Инференс (предсказание)** — это применение УЖЕ обученной модели к новым данным. Обучения здесь нет: загружаем 5 сохранённых моделей, каждую просим предсказать тест, 5 ответов **усредняем** (ансамбль) и записываем в CSV.

Почему усреднение? Каждая модель видела свои 4/5 данных и ошибается по-своему. Среднее 5 независимых ошибок стабильнее любого одиночного предсказания.

В ноутбуке повторим руками всё, что делает `predict.py`, и убедимся, что получается то же самое.

In [ ]:
# Переходим в корень проекта (если надо) и импортируем всё нужное.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import pandas as pd  # таблицы
import numpy as np  # математика
from pathlib import Path  # пути к файлам
# Импортируем ГОТОВУЮ функцию загрузки ансамбля из predict.py — не будем дублировать её код.
from predict import load_ensemble
# И наши функции обработки (та же обработка, что при обучении!).
from model_utils import generate_features, numeric_matrix, transform_features, ensure_columns, ID_COLUMN, TARGET_COLUMN
# Загружаем весь ансамбль из artifacts/: метаданные, признаки, препроцессоры, 5 моделей, моды.
metadata, feature_names, fold_medians, fold_means, fold_scales, models, modes = load_ensemble(Path("artifacts"))
print("Загружено моделей:", len(models))  # должно быть 5
print("Признаков:", len(feature_names))  # должен быть 61
print("Seed модели:", metadata["seed"])  # seed из метаданных пойдёт в имя файла

## 1. Готовим тест точно так же, как готовили train

In [ ]:
# Читаем открытый тест (в нём НЕТ колонки protection_score — её предсказываем).
test = pd.read_csv("hard_test.csv")
print("Тест:", test.shape)
# Шаг 1: те же 9 новых признаков, что и при обучении (порядок важен: сначала признаки!).
test = generate_features(test)
# Шаг 2: если какой-то колонки не хватает вообще — создать её из NaN (защита от кривого теста).
absent = ensure_columns(test, feature_names)
print("Отсутствующих колонок:", absent)  # пустой список = всё на месте
# Шаг 3: смотрим, сколько пропусков будет заполнено медианами (по каждой колонке).
nan_counts = test[feature_names].apply(pd.to_numeric, errors="coerce").isna().sum()
print(nan_counts[nan_counts > 0])  # показываем только колонки с пропусками
# Шаг 4: превращаем в сырую матрицу numpy (порядок колонок — feature_names, как при обучении!).
raw = numeric_matrix(test, feature_names)
print("Матрица теста:", raw.shape)

## 2. Предсказываем каждой моделью и усредняем

In [ ]:
# Готовим матрицу (5 моделей, 2000 строк) под предсказания каждого фолда.
fold_predictions = np.zeros((len(models), len(test)))
# enumerate даёт (номер фолда, модель). У каждого фолда — СВОЙ препроцессор!
for fold_index, model in enumerate(models):
    # Обрабатываем тест препроцессором ЭТОГО фолда (его медианы/средние/масштабы).
    transformed = transform_features(raw, fold_medians[fold_index], fold_means[fold_index], fold_scales[fold_index])
    # Предсказываем и кладём в строку матрицы.
    fold_predictions[fold_index] = model.predict(transformed)
print("Предсказания фолдов:", fold_predictions.shape)
# Смотрим на первые 3 клиентов: как различаются ответы 5 моделей.
print("Ответы 5 моделей для первых 3 клиентов:")
print(fold_predictions[:, :3].round(2))
# Усредняем по фолдам (axis=0 = схлопнуть строки-модели в одну строку-среднее).
ensemble = np.mean(fold_predictions, axis=0)
print("Среднее для первых 3 клиентов:", ensemble[:3].round(2))
# Разброс мнений моделей: std по фолдам. Где большой — там модели не уверены.
disagreement = fold_predictions.std(axis=0)
print(f"Средний разброс мнений моделей: {disagreement.mean():.3f}")

Обрати внимание: 5 моделей дают немного разные числа (например, 99.5 vs 100.3), а среднее — стабильный итог. Иногда предсказание чуть вылезает за [0, 100] — обрежем это дальше.

## 3. Обрезаем в [0, 100] и собираем submission

In [ ]:
# np.clip обрезает: всё, что < 0, становится 0; всё, что > 100, становится 100.
clipped = np.clip(ensemble, 0.0, 100.0)
print(f"До обрезки: {ensemble.min():.4f} .. {ensemble.max():.4f}")  # могло вылезти за края
print(f"После обрезки: {clipped.min():.4f} .. {clipped.max():.4f}")  # теперь строго внутри
# Собираем submission-таблицу: ID (как строка) + предсказание.
submission = pd.DataFrame({ID_COLUMN: test[ID_COLUMN].astype(str), TARGET_COLUMN: clipped})
print(submission.head())
# Сохраняем в /tmp (НЕ в корень проекта, чтобы не плодить файлы). float_format=%.4f — 4 знака.
submission.to_csv("/tmp/notebook_submission.csv", index=False, float_format="%.4f")
print("Сохранено: /tmp/notebook_submission.csv")

## 4. Проверяем submission как строгий жюри

In [ ]:
# Проверка 1: размер — 2000 строк, 2 колонки.
print("Размер:", submission.shape, "(ждём 2000x2)")
# Проверка 2: имена колонок — ровно customer_id и protection_score.
print("Колонки:", list(submission.columns))
# Проверка 3: пропусков нет (должно быть 0).
print("Пропусков:", int(submission.isna().sum().sum()))
# Проверка 4: все предсказания внутри [0, 100].
print("Всё в [0, 100]:", bool(submission[TARGET_COLUMN].between(0, 100).all()))
# Проверка 5: ID совпадают с тестом один в один и по порядку.
print("ID совпадают с тестом:", bool((submission[ID_COLUMN] == test[ID_COLUMN].astype(str)).all()))
# Проверка 6: ID уникальны (дублей нет).
print("ID уникальны:", submission[ID_COLUMN].nunique() == len(submission))

## 5. Сверяемся со скриптом predict.py: должно совпасть побитно

In [ ]:
# Запускаем настоящий predict.py и сравниваем его файл с нашим ручным.
import sys  # нужен ради sys.executable (путь к текущему Python)
!{sys.executable} predict.py --input-csv hard_test.csv --model-dir artifacts --output-csv /tmp/script_submission.csv
# Читаем файл скрипта.
script_submission = pd.read_csv("/tmp/script_submission.csv")
# np.allclose — поэлементное сравнение с tiny допуском (файлы писали с округлением до 4 знаков).
same = np.allclose(submission[TARGET_COLUMN], script_submission[TARGET_COLUMN], atol=1e-4)
print("Наш ручной результат совпадает с predict.py:", same)

## Выводы ноутбука 05

1. Инференс = загрузить 5 моделей → обработать тест → предсказать каждой → усреднить → обрезать → записать CSV.
2. Обработка теста — теми же функциями и параметрами, что при обучении (иначе мусор).
3. У каждого фолда свой препроцессор — не перепутай!
4. Submission проверяется 6 проверками выше (размер, колонки, NaN, диапазон, ID).
5. Наш ручной расчёт совпал с `predict.py` — значит, ты понял механику изнутри.

Дальше — **06_experiments**: сравниваем модели и отвечаем на вопрос «почему именно CatBoost?».